### 1. Inicialização (Ambiente cDCGAN)

In [ ]:
import torch
import torchvision.transforms as transforms
import torch.utils.data as data
import pandas as pd
import importlib
import optuna
import json
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from tqdm import tqdm

# Imports Modulares
import config as cfg
importlib.reload(cfg)

import dataset.dataloader as dl
importlib.reload(dl)

import utils.metrics as mtcs
importlib.reload(mtcs)

import utils.visualization as vis
importlib.reload(vis)

import generative.dcgan as gan
importlib.reload(gan)

import generative.tuning as tuning
importlib.reload(tuning)


# Configuração de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

cfg.GAN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.GAN_MODELS_DIR.mkdir(parents=True, exist_ok=True)
cfg.GAN_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.GAN_AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)

### 2. Preparação de Dados
Para garantir a validade científica da nossa comparação entre o cVAE e a cDCGAN, é imperativo que ambos os modelos sejam treinados exatamente com as mesmas imagens. Utilizamos a mesma semente de aleatoriedade (`RANDOM_SEED`) no particionamento dos dados.

In [ ]:
print("--- A carregar e a particionar os dados ---")

# 1. Carregamento e Preparação dos Dados
df = pd.read_csv(cfg.LABELS_PATH)

data_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Mantemos exatamente o mesmo split para não vazar dados
train_df, val_df, _ = dl.get_stratified_splits(
    df, test_size=cfg.TEST_SIZE, val_size=cfg.VAL_SIZE, random_state=cfg.RANDOM_SEED
)

train_dataset = dl.ButterflyDataset(df=train_df, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)
val_dataset = dl.ButterflyDataset(df=val_df, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)

# DataLoader (O Batch Size e o Shuffle mantêm-se iguais ao do VAE)
train_loader = data.DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = data.DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

n_classes = len(train_dataset.classes)
print(f"Total de classes condicional: {n_classes}")
print(f"Amostras de Treino para a GAN: {len(train_dataset)}")

### 3. Otimização de Hiperparâmetros (Optuna via KID Score)
Ao contrário dos VAEs, as redes GAN não possuem uma métrica de convergência baseada numa função de custo objetiva (a Loss indica apenas o equilíbrio de Nash). Portanto, parametrizamos o algoritmo TPE do Optuna para minimizar diretamente uma métrica de qualidade percetual gerativa: o *Kernel Inception Distance* (KID). Configuramos `epochs_trial` curtas para mapear rapidamente a estabilidade arquitetural.

In [ ]:
# Configuração do banco de dados do Optuna exclusivo para a GAN
OPTUNA_GAN_DB_PATH = cfg.GAN_RESULTS_DIR / "optuna_cgan_study.db"
STORAGE_URL_GAN = f"sqlite:///{OPTUNA_GAN_DB_PATH}" 

# Instanciar a Objective para a GAN
cgan_objective = tuning.CGANObjective(
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    n_classes=n_classes,
    img_size=cfg.IMAGE_SIZE,
    epochs_trial=10 # Épocas curtas para avaliação
)

# Criar e correr o estudo
study_cgan = optuna.create_study(
    study_name="cgan_optimization",
    direction='minimize', # Queremos minimizar o KID Score
    storage=STORAGE_URL_GAN, 
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

print(f"A iniciar a otimização de hiperparâmetros GAN (Base de Dados: {OPTUNA_GAN_DB_PATH})...")
study_cgan.optimize(cgan_objective, n_trials=15, show_progress_bar=True)

print("\n--- Melhores Hiperparâmetros GAN ---")
for key, value in study_cgan.best_trial.params.items():
    print(f"  {key}: {value}")

### 4. Treino Definitivo da cDCGAN
Instanciamos o Gerador e o Discriminador com os hiperparâmetros ótimos encontrados pelo Optuna. Executamos o treino completo, guardando o histórico das *Losses* de forma a podermos plotar posteriormente a evolução e verificar se a rede não sofreu de *Mode Collapse* ou desequilíbrio fatal (onde o discriminador aniquila o gerador prematuramente).

In [ ]:
# 1. Extrair os parâmetros vencedores do Optuna
best_latent_dim = study_cgan.best_trial.params['latent_dim']
best_lr = study_cgan.best_trial.params['lr']
best_embed = study_cgan.best_trial.params['embed_size']

print(f"Instanciando cDCGAN com latent_dim={best_latent_dim}, embed={best_embed} e lr={best_lr:.6f}")

# 2. Instanciar a Arquitetura
final_generator = gan.cDCGenerator(n_classes, best_latent_dim, 3, embed_size=best_embed).to(device)
final_discriminator = gan.cDCDiscriminator(n_classes, 3, img_size=cfg.IMAGE_SIZE).to(device)

final_generator.apply(gan.init_dcgan_weights)
final_discriminator.apply(gan.init_dcgan_weights)

# 3. Executar o Treino SOTA 
trained_generator, cgan_history = gan.train_cgan(
    generator=final_generator,
    discriminator=final_discriminator,
    loader=train_loader,
    latent_dim=best_latent_dim,
    epochs=cfg.N_EPOCHS,
    lr=best_lr,
    device=device,
    save_dir=cfg.GAN_MODELS_DIR
)

# 4. Avaliar a GAN 
mtcs.evaluate_gan(cgan_history, cfg.GAN_PLOTS_DIR)

### 5. Visualização Condicional (Grelha 1x4)
De forma semelhante à nossa *pipeline* no cVAE, extraímos amostras diretamente do espaço latente otimizado pela GAN, condicionando a geração às nossas 4 classes mais críticas. Este teste qualitativo imediato permite confirmar visualmente a superioridade fotorealista (ou os artefactos) gerados pelas camadas de convolução transposta em oposição ao método MSE.

In [ ]:
# 1. Carregar as classes críticas
target_df = pd.read_csv(cfg.TARGET_CLASSES_PATH)
top_4_classes = target_df['Classe'].tolist()[:4]

# 2. Gerar e guardar a grelha através do módulo de visualização
vis.generate_cgan_samples_grid(
    model=trained_generator,
    target_classes=top_4_classes,
    class_to_idx=train_dataset.class_to_idx,
    latent_dim=best_latent_dim,
    device=device,
    save_path=cfg.GAN_PLOTS_DIR / 'cgan_grid_1x4.png'
)

### 6. Avaliação Generativa Quantitativa (FID & KID)
Para quantificar o salto qualitativo da arquitetura GAN face ao VAE, extraímos *features* profundas utilizando a rede InceptionV3. O *Fréchet Inception Distance* (FID) e o *Kernel Inception Distance* (KID) avaliarão a distância entre a distribuição das imagens reais do conjunto de validação e a distribuição gerada pela nossa cDCGAN. Esperamos que a redução do desfoque traduza-se numa melhoria (redução) imediata destas métricas SOTA.

In [ ]:
print("A configurar o extrator de features (InceptionV3)...")
# normalize=True é crucial porque o nosso ToTensor() deixa as imagens em [0, 1] e não [0, 255]
fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
kid_metric = KernelInceptionDistance(feature=2048, subset_size=50, normalize=True).to(device)

trained_generator.eval()

print(f"A processar todo o Validation Loader para cálculo preciso de FID/KID...")
with torch.no_grad():
    for real_imgs, labels in tqdm(val_loader, desc="Calculando FID/KID in-memory"):
        real_imgs = real_imgs.to(device)
        labels = labels.to(device)
        batch_size = real_imgs.size(0)
        
        # 1. Atualizar métricas com imagens REAIS
        fid_metric.update(real_imgs, real=True)
        kid_metric.update(real_imgs, real=True)
        
        # 2. Gerar imagens FALSAS condicionadas às mesmas labels
        # (Usamos o best_latent_dim descoberto pelo Optuna na Célula 4)
        z = torch.randn(batch_size, best_latent_dim).to(device)
        
        # Na GAN usamos o forward direto em vez do decode()
        fake_imgs = trained_generator(z, labels)
        
        # 3. Atualizar métricas com imagens FALSAS
        fid_metric.update(fake_imgs, real=False)
        kid_metric.update(fake_imgs, real=False)

# Calcular os resultados finais
print("\nA extrair e computar a distância das distribuições...")
fid_score = fid_metric.compute()
kid_mean, kid_std = kid_metric.compute()

print("\n" + "="*40)
print(" RESULTADOS DA AVALIAÇÃO GENERATIVA (cDCGAN)")
print("="*40)
print(f"FID Score: {fid_score.item():.4f}")
print(f"KID Score: {kid_mean.item():.4f} ± {kid_std.item():.4f}")
print("="*40)

# Salvar métricas no disco 
metrics_summary = {
    "FID": fid_score.item(),
    "KID_mean": kid_mean.item(),
    "KID_std": kid_std.item()
}

with open(cfg.GAN_RESULTS_DIR / 'cgan_generative_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=4)

### 7. Geração em Massa (Pool de Candidatos cDCGAN)

Com a Rede Generativa Adversarial Condicional (cDCGAN) acabada de treinar, esta fase aproveita a velocidade do Gerador para esculpir rapidamente 500 imagens candidatas para cada classe crítica. Ao contrário do DDPM, o *forward pass* da GAN é quase instantâneo. O tensor resultante (cujos valores variam entre -1 e 1 devido à função de ativação Tanh) é automaticamente normalizado para a escala RGB [0, 255] e guardado no disco. O registo CSV gerado será a ponte para a fase final de *Hard Sample Mining*.

In [ ]:
import os
import pandas as pd
import torch
from torchvision.utils import save_image
from tqdm.auto import tqdm

# ==========================================
# 1. PARÂMETROS DA GERAÇÃO DE CANDIDATOS
# ==========================================
CANDIDATES_PER_CLASS = 500
BATCH_SIZE_GEN = 100

print(f"--- Fase 1: Geração em Massa cDCGAN (Pool de Candidatos) ---")
print(f"Alvo: Gerar {CANDIDATES_PER_CLASS} imagens adversárias por classe crítica.\n")

# Como o modelo acabou de ser treinado, colocamo-lo em modo eval.
final_generator.eval()

# ==========================================
# 2. CARREGAR CLASSES CRÍTICAS
# ==========================================
target_df = pd.read_csv(cfg.TARGET_CLASSES_PATH)
target_classes = target_df['Classe'].tolist()[:4]
print(f"Classes selecionadas para geração: {target_classes}\n")

candidates_data = []

with torch.no_grad():
    for class_name in target_classes:
        class_idx = train_dataset.class_to_idx[class_name]
        print(f"\nA criar {CANDIDATES_PER_CLASS} candidatos para [{class_name}]...")
        
        num_lotes = CANDIDATES_PER_CLASS // BATCH_SIZE_GEN
        
        for batch_idx, batch_start in enumerate(range(0, CANDIDATES_PER_CLASS, BATCH_SIZE_GEN)):
            current_batch_size = min(BATCH_SIZE_GEN, CANDIDATES_PER_CLASS - batch_start)
            print(f"   Gerando Lote {batch_idx + 1} de {num_lotes}...")
            
            # Prepara a label condicional
            labels = torch.full((current_batch_size,), class_idx, dtype=torch.long).to(device)
            
            # Gera o vetor latente otimizado 
            z = torch.randn(current_batch_size, best_latent_dim).to(device)
            
            # Gera as imagens via cDCGAN
            fake_imgs = final_generator(z, labels)
            
            # Salva no disco
            for j in range(current_batch_size):
                global_idx = batch_start + j
                safe_class_name = class_name.replace(' ', '_')
                img_filename = f"cgan_candidate_{safe_class_name}_{global_idx:03d}.jpg"
                img_path = cfg.GAN_AUGMENTED_DIR / img_filename
                
                save_image(fake_imgs[j], img_path, normalize=True, value_range=(-1, 1))
                
                candidates_data.append({
                    'filename': img_filename,
                    'label': class_name
                })

candidates_df = pd.DataFrame(candidates_data)
candidates_csv_path = cfg.GAN_RESULTS_DIR / "cgan_candidates_labels.csv"
candidates_df.to_csv(candidates_csv_path, index=False)

print(f"\nGeração de Candidatos cDCGAN Concluída!")
print(f"Total de {len(candidates_df)} imagens candidatas salvas na pasta: {cfg.GAN_AUGMENTED_DIR}")
print(f"Registo salvo em: {candidates_csv_path}")